## LAB02 Practica 1

In [1]:
## Importamos las librerias que vamos a utilizar

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random
import csv

1º Definimos los conjuntos de datos para las puertas lógicas AND y transformamos los datos a tensores.

In [2]:
# Fijar la semilla para pesos iniciales únicos en cada ejecución pero consistentes dentro de un experimento
random.seed()
torch.manual_seed(random.randint(0, 10000))

# Datos de entrada (X) y salida esperada (Y) para la puerta lógica AND
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = torch.tensor([[0.0], [0.0], [0.0], [1.0]])

def step_function(x):
    return 1.0 if x >= 0 else 0.0

# Generar pesos iniciales únicos por ejecución
initial_w = torch.tensor([random.uniform(-1, 1), random.uniform(-1, 1)], dtype=torch.float32)

# Función de entrenamiento
def train_perceptron(lr, b_init, max_epochs=100):
    w = initial_w.clone()
    b = torch.tensor(b_init, dtype=torch.float32)
    history = []  # Para almacenar todos los cambios
    
    for epoch in range(1, max_epochs + 1):
        outputs = []
        errors = []
        epoch_data = []  # Datos de la época
        
        for i in range(len(X)):
            w_initial = w.clone()  # Clonar pesos antes del ajuste
            net_input = torch.dot(w, X[i]) + b
            output = step_function(net_input)
            error = Y[i].item() - output
            errors.append(error)
            outputs.append(output)

            # Ajustar pesos
            w += lr * error * X[i]
            b += lr * error

            # Guardar datos de la iteración
            epoch_data.append((epoch, X[i].tolist(), Y[i].item(), w_initial.tolist(), output, error, w.tolist()))

        # Guardar la evolución de toda la época
        history.append(epoch_data)

        # Verificar si la salida es exactamente [0, 0, 0, 1] y errores son [0, 0, 0, 0]
        if outputs == [0, 0, 0, 1] and all(e == 0 for e in errors):
            return epoch, w, b, history  # Se detiene aquí
    
    return max_epochs, w, b, history  # Si no converge, retorna el máximo

# Pruebas con diferentes tasas de aprendizaje y valores de bias
learning_rates = [0.01, 0.05, 0.1, 0.25, 0.5]
bias_values = [-1.0, -0.5, 0.0, 0.5, 1.0]

best_lr = None
best_b = None
best_epochs = float('inf')
best_w = None
best_bias = None
best_history = None

# Crear el archivo CSV y escribir la cabecera
csv_filename = "L2P1-Perceptron.csv"
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Learning Rate", "Bias", "Epoch", "x1", "x2", "Yd", "w1 inicial", "w2 inicial", "Y", "Error", "w1 final", "w2 final"])

    print("{:<15} {:<10} {:<10}".format("Learning Rate", "Bias", "Epochs to Converge"))
    print("-" * 40)

    for lr in learning_rates:
        for b_init in bias_values:
            epochs, trained_w, trained_b, history = train_perceptron(lr, b_init)
            print("{:<15} {:<10} {:<10}".format(lr, b_init, epochs))
            
            # Guardar el mejor modelo
            if epochs < best_epochs:
                best_epochs = epochs
                best_lr = lr
                best_b = b_init
                best_w = trained_w
                best_bias = trained_b
                best_history = history

            # Escribir cada iteración en el CSV
            for epoch_data in history:
                for epoch, x_values, yd, w_initial, y, error, w_final in epoch_data:
                    writer.writerow([lr, b_init, epoch, int(x_values[0]), int(x_values[1]), int(yd),
                                    round(w_initial[0], 5), round(w_initial[1], 5),
                                    int(y), int(error),
                                    round(w_final[0], 5), round(w_final[1], 5)])

# Mostrar TODA la evolución de los pesos en cada iteración
print(f"\nMejor Learning Rate: {best_lr}, Mejor Bias: {best_b}, Épocas: {best_epochs}")
print("\nTabla de entrenamiento para el mejor LR y Bias:")
print("{:<6} {:<5} {:<5} {:<5} {:<10} {:<10} {:<5} {:<6} {:<10} {:<10}".format(
    "Epoch", "x1", "x2", "Yd", "w1 inicial", "w2 inicial", "Y", "Error", "w1 final", "w2 final"
))
print("-" * 90)

for epoch_data in best_history:
    for epoch, x_values, yd, w_initial, y, error, w_final in epoch_data:
        print("{:<6} {:<5} {:<5} {:<5} {:<10.2f} {:<10.2f} {:<5} {:<6} {:<10.2f} {:<10.2f}".format(
            epoch, int(x_values[0]), int(x_values[1]), int(yd),
            w_initial[0], w_initial[1], int(y), int(error), w_final[0], w_final[1]
        ))
    print("-" * 90)

print("\nResultados finales después del entrenamiento con los mejores parámetros:")
for i in range(len(X)):
    net_input = torch.dot(best_w, X[i]) + best_bias
    output = step_function(net_input)
    print(f'Entrada: {X[i].tolist()}, Salida: {output}')

print(f"\nLos datos han sido guardados en '{csv_filename}' correctamente.")

Learning Rate   Bias       Epochs to Converge
----------------------------------------
0.01            -1.0       55        
0.01            -0.5       44        
0.01            0.0        46        
0.01            0.5        62        
0.01            1.0        91        
0.05            -1.0       12        
0.05            -0.5       11        
0.05            0.0        14        
0.05            0.5        19        
0.05            1.0        25        
0.1             -1.0       7         
0.1             -0.5       10        
0.1             0.0        10        
0.1             0.5        12        
0.1             1.0        16        
0.25            -1.0       5         
0.25            -0.5       8         
0.25            0.0        8         
0.25            0.5        9         
0.25            1.0        10        
0.5             -1.0       7         
0.5             -0.5       7         
0.5             0.0        7         
0.5             0.5        8         
0